# install

In [1]:
# Cài đặt hoặc nâng cấp vnstock
!pip install -U vnstock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.8/275.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.5 MB/s eta 0:00:00


In [2]:
from vnstock import Quote
quote = Quote(symbol='ACB', source='KBS')


📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive


Ngày 8 — Deflated Sharpe Ratio (Bailey & López de Prado, 2014)

Mục tiêu:
1. Implement DSR đầy đủ (điều chỉnh skewness/kurtosis + số lượng trial N)
2. Áp dụng cho 12 factor đã build ở Ngày 4-5, trên universe 10 mã gốc — đặc biệt AlphaC_TrendCond
3. So sánh Sharpe thô vs t-stat (Sharpe/SE, chưa chỉnh N trials) vs DSR (đã chỉnh N=12 trials)

Câu hỏi cốt lõi: nếu tính đúng DSR ngay từ đầu (trên 10 mã), liệu nó có cảnh báo trước
việc AlphaC sẽ đảo dấu khi mở rộng lên 30 mã hay không — hay đây là lỗi mà chỉ multiple-testing
correction (không phải bản thân DSR) mới bắt được?

# Cell 1: Load dữ liệu

In [4]:
from vnstock import Quote
import pandas as pd
import numpy as np
import time
from scipy.stats import spearmanr, norm

SYMBOLS = [
    "ACB", "BID", "BSR", "CTG", "FPT", "GAS", "GVR", "HDB", "HPG", "LPB",
    "MBB", "MCH", "MSN", "MWG", "SAB", "SHB", "SSB", "SSI", "STB", "TCB",
    "TCX", "VCB", "VHM", "VIB", "VIC", "VJC", "VNM", "VPB", "VPL", "VRE"
]

START, END = "2024-01-01", "2025-12-31"
N_FWD = 5

raw = {}

for i, sym in enumerate(SYMBOLS, 1):

    # Nghỉ sau mỗi 18 request để tránh chạm limit 20/phút
    if i > 1 and (i - 1) % 18 == 0:
        print("⏳ Nghỉ 65 giây để tránh rate limit...")
        time.sleep(65)

    print(f"[{i}/{len(SYMBOLS)}] Loading {sym}...")

    try:
        df = Quote(symbol=sym, source="KBS").history(start=START,end=END,interval="d")

        raw[sym] = df.set_index("time")[["close", "open", "volume"]]

    except Exception as e:
        print(f"❌ {sym}: {type(e).__name__}")
        print(e)
        continue

price_df = pd.DataFrame({
    sym: d["close"] for sym, d in raw.items()
}).sort_index()

open_df = pd.DataFrame({
    sym: d["open"] for sym, d in raw.items()
}).sort_index()

volume_df = pd.DataFrame({
    sym: d["volume"] for sym, d in raw.items()
}).sort_index()

print(f"\n✅ Đã tải: {len(raw)}/{len(SYMBOLS)} mã")
print("Price shape:", price_df.shape)

price_df.head()


[1/30] Loading ACB...
[2/30] Loading BID...
[3/30] Loading BSR...
[4/30] Loading CTG...
[5/30] Loading FPT...
[6/30] Loading GAS...
[7/30] Loading GVR...
[8/30] Loading HDB...
[9/30] Loading HPG...
[10/30] Loading LPB...
[11/30] Loading MBB...
[12/30] Loading MCH...
[13/30] Loading MSN...
[14/30] Loading MWG...
[15/30] Loading SAB...
[16/30] Loading SHB...
[17/30] Loading SSB...
[18/30] Loading SSI...
⏳ Nghỉ 65 giây để tránh rate limit...
[19/30] Loading STB...
[20/30] Loading TCB...
[21/30] Loading TCX...
[22/30] Loading VCB...
[23/30] Loading VHM...
[24/30] Loading VIB...
[25/30] Loading VIC...
[26/30] Loading VJC...
[27/30] Loading VNM...
[28/30] Loading VPB...
[29/30] Loading VPL...
[30/30] Loading VRE...

✅ Đã tải: 30/30 mã
Price shape: (499, 30)


,ACB,BID,BSR,CTG,FPT,GAS,GVR,HDB,HPG,LPB,...,TCX,VCB,VHM,VIB,VIC,VJC,VNM,VPB,VPL,VRE
time,,,,,,,,,,,,,,,,,,,,,
2024-01-02 07:00:00,14.80,32.53,11.06,18.38,69.24,64.82,20.47,12.29,18.55,12.09,...,NaN,55.04,20.70,12.80,22.00,82.30,57.89,17.12,NaN,22.31
2024-01-03 07:00:00,15.13,33.13,11.12,18.65,69.53,65.17,21.10,12.35,18.78,12.24,...,NaN,55.70,20.90,12.99,22.08,82.84,58.49,17.35,NaN,22.45
2024-01-04 07:00:00,15.32,33.02,11.18,19.33,70.18,65.77,20.90,12.60,18.75,12.42,...,NaN,56.63,20.92,13.18,22.08,82.92,58.49,17.58,NaN,22.60
2024-01-05 07:00:00,15.41,33.66,11.24,19.60,70.32,66.19,21.30,12.66,18.78,12.50,...,NaN,56.82,20.75,13.38,22.05,82.76,58.32,17.44,NaN,22.55
2024-01-08 07:00:00,15.34,35.10,11.18,19.97,70.25,65.85,21.00,12.69,18.82,12.46,...,NaN,57.22,20.87,13.51,22.18,82.00,57.81,17.48,NaN,22.89


# Cell 2: các hàm


In [5]:
### Cell 2: Helper functions dùng để build factor
def rank(df: pd.DataFrame) -> pd.DataFrame:
    return df.rank(axis=1, pct=True)

def delta(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.diff(d)

def delay(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.shift(d)

def ts_sum(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).sum()

def ts_min(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).min()

def ts_max(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).max()

def ts_rank(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)

def correlation(df1: pd.DataFrame, df2: pd.DataFrame, d: int) -> pd.DataFrame:
    return df1.rolling(d).corr(df2)

def zlema(series: pd.Series, period: int = 20) -> pd.Series:
    lag = (period - 1) // 2
    adjusted = series + (series - series.shift(lag))
    return adjusted.ewm(span=period, adjust=False).mean()

def calc_rsi(price: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    delta_p = price.diff()
    gain = delta_p.clip(lower=0)
    loss = -delta_p.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)

# Cell 3: Các factor ngày 4

In [6]:
### Cell 3: 7 factor gốc (Ngày 4)
returns = price_df.pct_change(fill_method=None)
adv20 = volume_df.rolling(20).mean()

# 1. Momentum 12-1: return 12 tháng, bỏ tháng gần nhất
mom_12_1 = price_df.shift(21) / price_df.shift(252) - 1

# 2. Mean-reversion: z-score return 5 ngày so với phân phối 20 phiên gần nhất
ret_5d = price_df.pct_change(5, fill_method=None)
reversion_z = (ret_5d - ret_5d.rolling(20).mean()) / ret_5d.rolling(20).std()

# 3. Volume ratio: volume hôm nay / volume trung bình 20 phiên
volume_ratio = volume_df / adv20

# 4. Alpha A: đảo dấu momentum 3 ngày, nhân đồng biến open-volume 10 ngày
alpha_a = (-1 * rank(delta(returns, 3))) * correlation(open_df, volume_df, 10)

# 5. Alpha B: vị trí giá 10 phiên, độ cong giá, bất thường volume 5 phiên
alpha_b = ((-1 * rank(ts_rank(price_df, 10)))
           * rank(delta(delta(price_df, 1), 1))
           * rank(ts_rank(volume_df / adv20, 5)))

# 6. Alpha C: ternary theo chế độ thị trường (trend rõ vs sideway)
trend_cond = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)) <= 0.05
alpha_c = pd.DataFrame(
    np.where(trend_cond, -1 * (price_df - ts_min(price_df, 100)), -1 * delta(price_df, 3)),
    index=price_df.index, columns=price_df.columns,
)

# 7. ZLEMA mean-reversion: giá lệch khỏi ZLEMA-20 bao nhiêu %
zlema_df = price_df.apply(zlema)
alpha_zlema = -(price_df - zlema_df) / zlema_df

factors = {
    "Momentum_12_1": mom_12_1,
    "MeanReversion_Z": reversion_z,
    "VolumeRatio": volume_ratio,
    "AlphaA_RetCorr": alpha_a,
    "AlphaB_TSRankVol": alpha_b,
    "AlphaC_TrendCond": alpha_c,
    "ZLEMA_Reversion": alpha_zlema,
}

# Cell 4: Các factor ngày 5

In [7]:
### Cell 4: RSI-regime factors + ensemble với Alpha C (Ngày 5)
SLOPE_WINDOW = 5

rsi_df = calc_rsi(price_df, window=14)
rsi_slope = (rsi_df - rsi_df.shift(SLOPE_WINDOW)) / SLOPE_WINDOW

# AlphaD (No Regime): pure trend-following theo slope RSI
alpha_d = pd.DataFrame(np.sign(rsi_slope), index=rsi_df.index, columns=rsi_df.columns)

# AlphaD4 (Regime Switch): override reversal khi RSI chạm cực trị 100 ngày
rsi_high_100 = rsi_df >= ts_max(rsi_df, 100)
rsi_low_100 = rsi_df <= ts_min(rsi_df, 100)
alpha_d4 = pd.DataFrame(
    np.select([rsi_high_100.values, rsi_low_100.values], [-1, 1], default=alpha_d.values),
    index=rsi_df.index, columns=rsi_df.columns,
)

# AlphaD5 (Soft Regime): blend liên tục theo cường độ trend + percentile RSI
trend_strength = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)).abs()
trend_weight = np.tanh(trend_strength / 0.05)

rsi_range_100 = ts_max(rsi_df, 100) - ts_min(rsi_df, 100)
rsi_pct_pos = (rsi_df - ts_min(rsi_df, 100)) / rsi_range_100
extreme_high = rsi_pct_pos >= 0.9
extreme_low = rsi_pct_pos <= 0.1

slope_signal_continuous = np.tanh(rsi_slope / 2)
blended_signal = trend_weight * slope_signal_continuous
alpha_d5 = pd.DataFrame(
    np.select([extreme_high.values, extreme_low.values], [-1, 1], default=blended_signal.values),
    index=rsi_df.index, columns=rsi_df.columns,
)

# Ensemble: Alpha C (rank-scaled [-1,1]) trung bình 50/50 với D4 / D5
alpha_c_scaled = 2 * rank(alpha_c) - 1
ensemble_c_d4 = (alpha_c_scaled + alpha_d4) / 2
ensemble_c_d5 = (alpha_c_scaled + alpha_d5) / 2

factors.update({
    "AlphaD_NoRegime": alpha_d,
    "AlphaD4_RegimeSwitch": alpha_d4,
    "AlphaD5_SoftRegime": alpha_d5,
    "Ensemble_C_D4": ensemble_c_d4,
    "Ensemble_C_D5": ensemble_c_d5,
})

print(f"Tổng số factor: {len(factors)}")
list(factors.keys())

Tổng số factor: 12


['Momentum_12_1',
 'MeanReversion_Z',
 'VolumeRatio',
 'AlphaA_RetCorr',
 'AlphaB_TSRankVol',
 'AlphaC_TrendCond',
 'ZLEMA_Reversion',
 'AlphaD_NoRegime',
 'AlphaD4_RegimeSwitch',
 'AlphaD5_SoftRegime',
 'Ensemble_C_D4',
 'Ensemble_C_D5']

# Cell 5: IC cho các factor

In [8]:
### Cell 5: Tính IC (Spearman, cross-sectional theo ngày) cho toàn bộ factor
fwd_ret = price_df.shift(-N_FWD) / price_df - 1

def calc_ic(factor_df: pd.DataFrame, fwd_ret_df: pd.DataFrame, min_valid: int = 5) -> pd.Series:
    """Cross-sectional Spearman IC giữa factor và forward return, theo từng ngày."""
    ic = {}
    for date in factor_df.index:
        f, r = factor_df.loc[date], fwd_ret_df.loc[date]
        valid = f.notna() & r.notna()
        if valid.sum() < min_valid:
            continue
        f_valid, r_valid = f[valid], r[valid]
        if f_valid.nunique() < 2 or r_valid.nunique() < 2:
            continue  # tránh warning/NaN khi factor hoặc forward return constant trong ngày đó
        ic[date] = spearmanr(f_valid, r_valid)[0]
    return pd.Series(ic).dropna()

ic_series = {name: calc_ic(f, fwd_ret) for name, f in factors.items()}

# Align về cùng tập ngày quan sát (giao của index) — công bằng khi so sánh giữa các factor
common_idx = None
for s in ic_series.values():
    common_idx = s.index if common_idx is None else common_idx.intersection(s.index)
ic_common = {name: s.loc[common_idx] for name, s in ic_series.items()}

def ic_stats(s: pd.Series) -> dict:
    return {
        "Mean IC": s.mean(), "Std IC": s.std(),
        "IC IR": s.mean() / s.std(),
        "t-stat": s.mean() / s.std() * np.sqrt(len(s)),
        "Hit Rate": (s > 0).mean(), "N Obs": len(s),
    }

summary = pd.DataFrame({name: ic_stats(s) for name, s in ic_common.items()}).T.round(4)
print(f"Common obs: {len(common_idx)}")
summary

Common obs: 235


,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
Momentum_12_1,0.0154,0.3101,0.0497,0.7621,0.5021,235.0
MeanReversion_Z,0.0093,0.2318,0.0400,0.6127,0.5021,235.0
VolumeRatio,0.0328,0.1988,0.1652,2.5318,0.6000,235.0
AlphaA_RetCorr,0.0208,0.1975,0.1053,1.6147,0.5447,235.0
AlphaB_TSRankVol,-0.0359,0.2104,-0.1708,-2.6180,0.4468,235.0
AlphaC_TrendCond,-0.0087,0.2234,-0.0389,-0.5964,0.4809,235.0
ZLEMA_Reversion,-0.0383,0.2314,-0.1655,-2.5368,0.4170,235.0
AlphaD_NoRegime,0.0099,0.2155,0.0459,0.7043,0.5319,235.0
AlphaD4_RegimeSwitch,-0.0120,0.2171,-0.0552,-0.8461,0.4809,235.0
AlphaD5_SoftRegime,-0.0264,0.2390,-0.1107,-1.6963,0.4596,235.0


# Cell 6: DSR


Deflated Sharpe Ratio (Bailey & López de Prado, 2014)

Coi chuỗi IC hàng ngày của mỗi factor như "return" của một chiến lược (đúng cách IC IR đã
được dùng như Sharpe ratio của tín hiệu xuyên suốt project). DSR trả lời: "xác suất Sharpe
THẬT lớn hơn mức Sharpe cao nhất kỳ vọng đạt được thuần túy do may rủi, nếu đã thử N chiến
lược độc lập" — gộp 2 điều chỉnh:

- **Non-normality**: skewness/kurtosis thật của chuỗi IC (không giả định normal).
- **Multiple testing**: benchmark so sánh là `SR_0*` (Sharpe max kỳ vọng dưới null với N trials),
  không phải 0.

In [9]:
### Cell 6: Deflated Sharpe Ratio — implementation đầy đủ
EULER_MASCHERONI = 0.5772156649015329

def sharpe_std_error(sr: float, skew: float, kurt: float, n_obs: int) -> float:
    """SE(SR_hat), điều chỉnh skew/kurtosis (Bailey & López de Prado, 2012).
    SE(SR) = sqrt( (1 - skew*SR + (kurt-1)/4 * SR^2) / (T-1) )
    kurt: kurtosis kiểu Pearson (normal=3), KHÔNG phải excess kurtosis.
    """
    return np.sqrt((1 - skew * sr + (kurt - 1) / 4 * sr ** 2) / (n_obs - 1))


def expected_max_sharpe(sr_trials: np.ndarray, n_trials: int) -> float:
    """SR_0*: Sharpe tối đa kỳ vọng thuần túy do may rủi nếu thử n_trials chiến lược độc lập,
    không chiến lược nào có skill thật (null hypothesis).
    SR_0* = sqrt(Var[SR_n]) * [(1-γ)*Z^-1(1-1/N) + γ*Z^-1(1-1/(N*e))]
    """
    var_sr = np.var(sr_trials, ddof=1)
    sr_std = np.sqrt(var_sr)
    return sr_std * (
        (1 - EULER_MASCHERONI) * norm.ppf(1 - 1 / n_trials)
        + EULER_MASCHERONI * norm.ppf(1 - 1 / (n_trials * np.e))
    )


def probabilistic_sharpe_ratio(returns: pd.Series, sr_benchmark: float = 0.0) -> dict:
    """PSR(SR*): xác suất Sharpe thật > benchmark, chỉnh skew/kurtosis, CHƯA chỉnh multiple testing."""
    n_obs = len(returns)
    sr_hat = returns.mean() / returns.std()
    skew = returns.skew()
    kurt = returns.kurtosis() + 3  # pandas .kurtosis() = excess kurtosis -> +3 về Pearson kurtosis

    se_sr = sharpe_std_error(sr_hat, skew, kurt, n_obs)
    z = (sr_hat - sr_benchmark) / se_sr
    return {"SR_hat": sr_hat, "Skewness": skew, "Kurtosis": kurt,
            "SE(SR_hat)": se_sr, "Z-score": z, "PSR": norm.cdf(z), "N_obs": n_obs}


def deflated_sharpe_ratio(returns: pd.Series, sr_trials: np.ndarray, n_trials: int) -> dict:
    """DSR = PSR(SR_0*): benchmark là SR_0* thay vì 0.
    DSR ~ 1   -> Sharpe vượt xa mức may rủi kỳ vọng trong N lần thử -> skill đáng tin.
    DSR ~ 0.5 -> không phân biệt được với kết quả ngẫu nhiên tốt nhất trong N lần thử.
    """
    n_obs = len(returns)
    sr_hat = returns.mean() / returns.std()
    skew = returns.skew()
    kurt = returns.kurtosis() + 3

    sr_0 = expected_max_sharpe(sr_trials, n_trials)
    se_sr = sharpe_std_error(sr_hat, skew, kurt, n_obs)
    z = (sr_hat - sr_0) / se_sr

    return {"SR_hat": sr_hat, "Skewness": skew, "Kurtosis": kurt,
            "SR_0_star": sr_0, "SE(SR_hat)": se_sr, "Z-score": z,
            "DSR": norm.cdf(z), "N_obs": n_obs, "N_trials": n_trials}

# Cell 7: Check SR0

In [10]:
### Cell 9: Sanity check — DSR trên chuỗi random thuần (kỳ vọng DSR ~ 0.5, không có skill thật)
rng = np.random.default_rng(0)
fake_returns = pd.Series(rng.normal(0, 1, 250))
fake_trials = rng.normal(0, 0.15, 12)

print(deflated_sharpe_ratio(fake_returns, sr_trials=fake_trials, n_trials=12))

{'SR_hat': np.float64(-0.00580996468069035), 'Skewness': np.float64(-0.018074045091663046), 'Kurtosis': np.float64(2.9262689551813943), 'SR_0_star': np.float64(0.24391482343655813), 'SE(SR_hat)': np.float64(0.06336961271417571), 'Z-score': np.float64(-3.940765572351136), 'DSR': np.float64(4.061098667464669e-05), 'N_obs': 250, 'N_trials': 12}


# Cell 8: DSR cho các factor

In [11]:
### Cell 8: DSR cho toàn bộ 12 factor, N_trials=12 (đúng số factor đã test trong project)
N_TRIALS = 12
sr_trials = summary["IC IR"].values  # dùng Sharpe (IC IR) của cả 12 factor để ước lượng Var[SR_n]

dsr_results = {
    name: deflated_sharpe_ratio(ic_s, sr_trials=sr_trials, n_trials=N_TRIALS)
    for name, ic_s in ic_common.items()
}
dsr_table = pd.DataFrame(dsr_results).T.round(4)
dsr_table.sort_values("DSR", ascending=False)

,SR_hat,Skewness,Kurtosis,SR_0_star,SE(SR_hat),Z-score,DSR,N_obs,N_trials
VolumeRatio,0.1652,-0.2397,3.0559,0.1725,0.0671,-0.1097,0.4563,235.0,12.0
AlphaA_RetCorr,0.1053,-0.0805,2.4029,0.1725,0.0658,-1.0215,0.1535,235.0,12.0
Momentum_12_1,0.0497,-0.0888,2.5714,0.1725,0.0655,-1.8735,0.0305,235.0,12.0
AlphaD_NoRegime,0.0459,-0.1827,2.5701,0.1725,0.0657,-1.9274,0.0270,235.0,12.0
MeanReversion_Z,0.0400,-0.0529,2.4735,0.1725,0.0655,-2.0250,0.0214,235.0,12.0
AlphaC_TrendCond,-0.0389,0.0411,2.9305,0.1725,0.0654,-3.2304,0.0006,235.0,12.0
Ensemble_C_D4,-0.0430,-0.2585,3.1949,0.1725,0.0650,-3.3137,0.0005,235.0,12.0
Ensemble_C_D5,-0.0527,0.0105,2.7778,0.1725,0.0654,-3.4421,0.0003,235.0,12.0
AlphaD4_RegimeSwitch,-0.0552,-0.2741,3.0461,0.1725,0.0649,-3.5073,0.0002,235.0,12.0
AlphaB_TSRankVol,-0.1708,0.0350,2.7764,0.1725,0.0660,-5.2025,0.0000,235.0,12.0


Kết quả bị đảo ngược hoàn toàn khi chiến lược alpha C giờ đây có DSR ko đáng tin cậy => các mô hình ensemble cũng bị mất đi giá trị

Volume ratio trở thành chiến lược có DSR cao nhất dù chưa vượt được ngưỡng 0.5

# Cell 9: Sharpe thô vs t-stat vs DSR

In [12]:
### Cell 9: Sharpe thô vs t-stat (Sharpe/SE, chưa chỉnh N) vs DSR (chỉnh N=12 trials)
compare_3way = pd.DataFrame({
    "Sharpe_tho (IC_IR)": summary["IC IR"],
    "t_stat (chua chinh N trials)": summary["t-stat"],
    "DSR_Zscore (N=12)": dsr_table["Z-score"],
    "DSR (0-1)": dsr_table["DSR"],
}).round(4).sort_values("DSR (0-1)", ascending=False)

compare_3way

,Sharpe_tho (IC_IR),t_stat (chua chinh N trials),DSR_Zscore (N=12),DSR (0-1)
VolumeRatio,0.1652,2.5318,-0.1097,0.4563
AlphaA_RetCorr,0.1053,1.6147,-1.0215,0.1535
Momentum_12_1,0.0497,0.7621,-1.8735,0.0305
AlphaD_NoRegime,0.0459,0.7043,-1.9274,0.0270
MeanReversion_Z,0.0400,0.6127,-2.0250,0.0214
AlphaC_TrendCond,-0.0389,-0.5964,-3.2304,0.0006
Ensemble_C_D4,-0.0430,-0.6592,-3.3137,0.0005
Ensemble_C_D5,-0.0527,-0.8079,-3.4421,0.0003
AlphaD4_RegimeSwitch,-0.0552,-0.8461,-3.5073,0.0002
AlphaB_TSRankVol,-0.1708,-2.6180,-5.2025,0.0000


Các chiến lược vô nghĩa ở dữ liệu 10 mã, lại trở nên có ý nghĩa hơn ở dữ liệu 30 mã, có thể ảnh hưởng là do các cổ phiếu VIC - VHM - VRE - VPL là những cổ phiếu có vốn hóa lớn, ảnh hưởng mạnh đến thị trường

Các cp này thường xuyên hút dòng tiền của các nhóm khác => dẫn đến kết quả bị biến động so với dữ liệu 10 mã (vì dữ liệu 10 mã chỉ có 1 cổ phiếu thuộc nhóm trên là VRE)

# Cell 10: Nhìn kỹ alpha C


In [13]:
### Cell 10: Zoom riêng AlphaC_TrendCond — nhân vật chính của câu chuyện "đảo dấu ở 30 mã"
print("=== AlphaC_TrendCond, universe 10 mã ===")
print(compare_3way.loc["AlphaC_TrendCond"])
print()
print(f"DSR median toàn bộ 12 factor: {dsr_table['DSR'].median():.4f}")
print(f"DSR AlphaC_TrendCond:         {dsr_table.loc['AlphaC_TrendCond', 'DSR']:.4f}")
print(f"Rank DSR của AlphaC trong {N_TRIALS} factor: "
      f"{(dsr_table['DSR'] > dsr_table.loc['AlphaC_TrendCond', 'DSR']).sum() + 1} / {N_TRIALS}")

=== AlphaC_TrendCond, universe 10 mã ===
Sharpe_tho (IC_IR)             -0.0389
t_stat (chua chinh N trials)   -0.5964
DSR_Zscore (N=12)              -3.2304
DSR (0-1)                       0.0006
Name: AlphaC_TrendCond, dtype: float64

DSR median toàn bộ 12 factor: 0.0005
DSR AlphaC_TrendCond:         0.0006
Rank DSR của AlphaC trong 12 factor: 6 / 12


Sharpe âm và DSR ko có ý nghĩa => chiến lược thất bại với dữ liệu mới